# denoise-audio
Requires python 3.10

Calls methods from denoise-audio

https://pypi.org/project/denoise-audio/

https://pypistats.org/packages/denoise-audio

Package came out Feb 2026 and has been updated 3x


Less commonly used but one of the few that works out of the box (don't need to download a C library) on .wav files and has fewer dependency conflicts

Uses 3 neural network backends trained to remove background noise:
 - rnnoise (fast, CPU-only, low-latency)
 - deepfilternet (high-quality full-band denoising)
 - fbdenoiser (FacebookResearch Denoiser / causal Demucs; strong enhancement)

I found that rnnoise does not work due to dependency conflict. But DeepFilterNet and Facebook Denoiser do work. Facebook Research Denoiser needs an ssl and downloads the model on the first run

In [1]:
%pip install denoise-audio

Looking in indexes: https://nexus.cainc.com/repository/pypi/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 6.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.8/49.8 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 6.6 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.2/113.2 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 MB 19.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 11.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 11.1 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
import os
import sys
from pathlib import Path
import time
from denoise import denoise_file

In [2]:
# have to modify where notebook looks for imports so it looks in the project root, not the notebook folder
os.getcwd()
sys.path.append(str(Path().resolve().parent))
print(sys.path)

['/Library/Frameworks/Python.framework/Versions/3.10/lib/python310.zip', '/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10', '/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/lib-dynload', '', '/Users/KRubio/CA_Repos/kays_repos/audio_quality_classification/.venv310/lib/python3.10/site-packages', '/Users/KRubio/CA_Repos/kays_repos/audio_quality_classification']


In [3]:
# Check which models work
for model in ["rnnoise", "deepfilternet", "fbdenoiser"]:
    try:
        for _ in denoise_file("input.wav", f"output_{model}.wav", model=model):
            pass
        print(f"{model}: ✓ works")
    except Exception as e:
        print(f"{model}: ✗ {e}")

rnnoise: ✗ pyrnnoise is not available: No module named 'av.option'
deepfilternet: ✗ input.wav
fbdenoiser: ✗ input.wav


In [25]:
# Choose one: rnnoise | deepfilternet | fbdenoiser
MODEL = "fbdenoiser"

filename = 'write'
input_filepath = f'../data/audio/{filename}.wav'
output_filepath = f'../data/audio/{filename}_denoise_{MODEL}.wav'

In [5]:
# Need for FacebookResearch Denoiser to download model, only use in development
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

In [26]:
# Model-specific kwargs (examples below)
kwargs = {}

# Example: RNNoise
# kwargs = {"rnnoise_sample_rate": 48000}

# Example: DeepFilterNet
# kwargs = {"df_model": "DeepFilterNet3", "df_pf": True, "df_compensate_delay": True}

# Example: FacebookResearch Denoiser
# kwargs = {"fb_model": "dns64", "fb_device": "cpu", "fb_dry": 1.0}

start = time.time()
for _ in denoise_file(input_filepath, output_filepath, model=MODEL, **kwargs):
    pass

print(f"elapsed: {time.time() - start:.2f}s")
print(f"Wrote: {output_filepath}")


elapsed: 3.12s
Wrote: ../data/audio/write_denoise_fbdenoiser.wav
